In [1]:
import pandas as pd
import re

In [2]:
# Cargamos los datos

data = pd.read_csv("processed_dataset.csv", low_memory=False)

In [3]:
# Limpieza de datos

data = data.dropna(subset=["post", "clean_post"])   # Eliminar valores faltantes

In [4]:
data.post

0         There's a lot of reasons to want to know all t...
1         I'm cross posting this from /r/cyberlaw, hopef...
2         DotCed,a Functional Analytics Consultant, offe...
3         Here is the program details of the data analyt...
4         i decided grad school (physics) was not for me...
                                ...                        
274234    Hello! I am wonder how the following output wo...
274235    Can anybody recommend me a book with medical s...
274236    I'm trying to build a simple Markov chain. I h...
274237    How do I view all available Rcpp::plugins? Tha...
274238    Hi everybody,\n\nI am currently doing a factor...
Name: post, Length: 272240, dtype: object

##### Encontrar los subreddits

In [5]:
reddit_exp = re.compile(r"\/r\/[a-z]+")

def find_subreddit_mentions(text: str) -> list:
    """Encuentra y devuelve todas las menciones de subreddits en el texto dado."""
    
    subreddit = reddit_exp.findall(text)
    return subreddit

In [6]:
# Prueba

find_subreddit_mentions(data.post[1])

['/r/cyberlaw']

##### Extraer las URLs

In [7]:
url_exp = re.compile(r"http[s]?\:\/\/\S+")

def url_extraction(text: str) -> list:
    """Extrae todas las URLs del texto dado."""
    
    urls = url_exp.findall(text)
    return urls

In [8]:
# Prueba

url_extraction(data.post[1])

['http://ad.doubleclick.net/click;h=(junk',
 'http://googleads.g.doubleclick.net/aclk?sa=l&amp;ai=(junk',
 'http://www.papajohns.com/index.shtm?utm_source=googlenetwork&amp;utm_medium=DisplayCPC&amp;utm_campaign=GoogleRemarketing',
 'http://googleads.g.doubleclick.net).']

##### Extraer los números de teléfono

In [9]:
phone_exp = re.compile(r"\+?[0-9]{9,10}|\([0-9]{3}\)\s?[0-9]{3}\-[0-9]{3,4}|[0-9]{3}[\-\s\_][0-9]{3}[\-\s\_][0-9]{3,4}")

def phone_number_extraction(text: str) -> list:
    """Extrae todos los números de teléfono del texto dado."""

    phones = phone_exp.findall(text)
    return phones

In [10]:
# Ejemplo

text = "Call me at +123456789 or (123) 456-7890. Alternatively, try 123-456-7890 or +1987654321."

phone_number_extraction(text)

['+123456789', '(123) 456-7890', '123-456-7890', '+1987654321']

##### Extraer las fechas

In [11]:
from datetime import datetime

datetime.fromisoformat('2011-11-04')

datetime.datetime(2011, 11, 4, 0, 0)

In [12]:
data.post[data.post.str.contains("2024")]

3938      According to a new report Global In-store Anal...
18879     I am a grad student at U of T and taking a [gr...
23178     I want to visualize the following transition m...
34362     Each of these dates are when the planet Mars w...
37131     Hi. I'm an undergrad who's currently taking a ...
47931      Hey, I’m Alex Miller, Director of Data Scienc...
49040     Hello,  \n\n\nI am a recent B.Sc Hons Psycholo...
51367     Hi all,\n\nI'm a sophomore at the University o...
52004     Hello, so I am struggling to figure out how to...
52099     Hello, so I have tried using the curve\_fit() ...
63878     Hello frens,\n\n&amp;#x200B;\n\nI uploaded a [...
66684     I would like to plan a practical self study ML...
74926     I have fed an LSTM  the number of students in ...
77857     I am using a  faster-rcnn (resnet-101) from te...
94622     Hello, so I am struggling to figure out how to...
94669     Hello, so I have tried using the curve\_fit() ...
113494     Saw this post trending on twi

In [13]:
data.post[data.post.str.contains("2024")][3938]

'According to a new report Global In-store Analytics Market, published by KBV Research, The Global In-store Analytics Market size is expected to reach $3.3 billion by 2024, rising at a market growth of 21.6% CAGR during the forecast period.\n\n&amp;#x200B;\n\n![img](0b0n0a4ti8b31 "In store Analytics")\n\nThe North America market would dominate the Global In-Store Analytics Software Market by Region by 2024, growing at a CAGR of 19.1 % during the forecast period. The Europe market is expected to witness a CAGR of 19.6% during (2018-2024). Additionally, The Asia Pacific market is expected to witness a CAGR of 22.7% during (2018-2024).\n\nThe Large Enterprises market dominated the Global In-Store Analytics Market by Organization Size 2017. The Small &amp; Medium-Sized Enterprises market is expected to witness highest CAGR of 23.4% during (2018-2024).\n\nThe On-Premise market dominated the Global In-Store Analytics Market by Deployment Mode 2017, growing at a CAGR of 19.9 % during the fore

In [14]:
import nltk
from nltk.tokenize import word_tokenize

# nltk.download("punkt")   # Descargar el paquete de tokenización de NLTK si es necesario

def dates_extraction_nlp(text: str) -> list:
    """Extrae y devuelve todas las fechas válidas en el texto dado utilizando NLP."""
    
    # Tokenizar el texto en palabras
    words = word_tokenize(text)
    valid_dates = []
    
    # Iterar sobre cada palabra tokenizada
    for word in words:
        try:
            # Intentar convertir la palabra en una fecha
            parsed_date = pd.to_datetime(word, errors="raise", dayfirst=False)
            valid_dates.append(parsed_date)
        except Exception:
            # Si la conversión falla, continuar con la siguiente palabra
            continue
    
    return valid_dates

# Ejemplo
dates = dates_extraction_nlp(data.post[3938])
for date in dates:
    # Imprimir las fechas válidas encontradas en formato "día-mes-año"
    print((date, date.strftime("%d-%m-%Y")))

(Timestamp('2024-01-01 00:00:00'), '01-01-2024')
(Timestamp('2024-01-01 00:00:00'), '01-01-2024')
(Timestamp('2017-01-01 00:00:00'), '01-01-2017')
(Timestamp('2017-01-01 00:00:00'), '01-01-2017')


In [15]:
import spacy
import dateparser

def extract_dates(text):
    """Función para extraer y analizar fechas de un texto dado utilizando spaCy y dateparser.

    - return: Una lista de tuplas con la fecha original y la fecha procesada.
    """
    
    # Cargar el modelo de lenguaje de spaCy
    nlp = spacy.load("en_core_web_sm")
    
    # Procesar el texto con el modelo de spaCy para extraer las entidades
    doc = nlp(text)
    
    dates = []
    
    # Iterar sobre las entidades identificadas por spaCy
    for ent in doc.ents:
        # Comprobar si la entidad es una fecha o una hora
        if ent.label_ == "DATE" or ent.label_ == "TIME":
            # Usar dateparser para analizar la fecha y convertirla a un formato estandarizado
            date = dateparser.parse(ent.text, settings={"DATE_ORDER": "DMY"})
            # Añadir la fecha original y la fecha procesada a la lista
            dates.append((ent.text, date))
    
    return dates


# Ejemplo
text = """
The conference will be held on March 9, 2023. John's birthday party is on 05/12/2023.
The meeting scheduled for next Wednesday has been postponed.
"""

# Llamar a la función y mostrar los resultados
extracted_dates = extract_dates(text)
for original, parsed in extracted_dates:
    print(f"Date found: {original}, Parsed as: {parsed}")

Date found: March 9, 2023, Parsed as: 2023-03-09 00:00:00
Date found: next Wednesday, Parsed as: None


##### Extraer códigos de programación o HTML

In [16]:
data.columns

Index(['created_date', 'created_timestamp', 'subreddit', 'title', 'author',
       'author_created_utc', 'full_link', 'score', 'num_comments',
       'num_crossposts', 'subreddit_subscribers', 'post', 'sentiment',
       'clean_post'],
      dtype='object')

##### Extracción de datos de APIs para entrenar un modelo

In [17]:
# import requests

# def extract_code(text):
#     # Expresión regular para detectar bloques de código en HTML (etiquetas <pre> y <code>)
#     code_block_pattern = r"<pre><code>(.*?)</code></pre>"
    
#     # Extraer bloques de código
#     code_blocks = re.findall(code_block_pattern, text, re.DOTALL)
    
#     # Combinar todos los fragmentos de código encontrados
#     code_snippets = " ".join(code_blocks)
    
#     return code_snippets

# def is_code(text):
#     return "code" if extract_code(text) else "non-code"

# # Obtener Datos de StackOverflow
# url_stackoverflow = "https://api.stackexchange.com/2.2/questions"
# params_stackoverflow = {
#     "site": "stackoverflow",
#     "tagged": "python",
#     "filter": "withbody",  # Para obtener el cuerpo de la pregunta
#     "pagesize": 100,       # Número de preguntas por página
#     "page": 1              # Página inicial
# }

# all_data_stackoverflow = []

# # Obtener datos de múltiples páginas de StackOverflow
# for page in range(1, 26):   # 100 páginas * 100 preguntas por página = 10,000 preguntas
#     params_stackoverflow["page"] = page
#     try:
#         response = requests.get(url_stackoverflow, params=params_stackoverflow)
#         response.raise_for_status()   # Verificar si la solicitud fue exitosa
#         data = response.json()
#         if "items" in data:
#             all_data_stackoverflow.extend(data["items"])
#             print(f"StackOverflow Page {page} processed")
#         else:
#             print(f"StackOverflow Page {page} does not contain 'items'")
#     except requests.exceptions.RequestException as e:
#         print(f"Error processing StackOverflow page {page}: {e}")
#     except ValueError as e:
#         print(f"Error decoding JSON for StackOverflow page {page}: {e}")

# # Procesar los Datos de StackOverflow
# texts_stackoverflow = []
# labels_stackoverflow = []

# for item in all_data_stackoverflow:
#     body = item.get("body", "")
#     code_snippets = extract_code(body)
#     if code_snippets:
#         texts_stackoverflow.append(code_snippets)
#         labels_stackoverflow.append("code")
#     else:
#         texts_stackoverflow.append(body)
#         labels_stackoverflow.append("non-code")

# # Crear el DataFrame de StackOverflow
# df_stackoverflow = pd.DataFrame({
#     "text": texts_stackoverflow,
#     "label": labels_stackoverflow
# })

# # Guardar el DataFrame en un archivo CSV
# df_stackoverflow.to_csv("stackoverflow_code_dataset.csv", index=False)

# # Mostrar las primeras filas del DataFrame
# print(df_stackoverflow.head())

In [18]:
# import pandas as pd
# import requests
# import base64

# def extract_github_code_snippets(query, language="Python", max_results=10):
#     """
#     Busca repositorios en GitHub y extrae fragmentos de código de README.md.
#     """
#     def get_file_content(owner, repo, path):
#         url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
#         headers = {"Accept": "application/vnd.github.v3+json"}
#         response = requests.get(url, headers=headers)
#         if response.status_code == 200:
#             file_data = response.json()
#             if file_data.get("encoding") == "base64":
#                 return base64.b64decode(file_data["content"]).decode("utf-8")
#         return None

#     url = "https://api.github.com/search/repositories"
#     headers = {"Accept": "application/vnd.github.v3+json"}
#     params = {"q": f"{query} language:{language}", "sort": "stars", "per_page": max_results}

#     response = requests.get(url, headers=headers, params=params)
#     repos = response.json().get("items", [])

#     texts = []
#     labels = []

#     for repo in repos:
#         owner = repo["owner"]["login"]
#         name = repo["name"]
#         try:
#             content = get_file_content(owner, name, "README.md")
#             if content:
#                 texts.append(content)
#                 labels.append("code")
#             else:
#                 texts.append(f"Repo {name} has no README.md")
#                 labels.append("non-code")
#         except Exception as e:
#             texts.append(str(e))
#             labels.append("error")

#     df = pd.DataFrame({"text": texts, "label": labels})
#     df.to_csv("github_code_dataset_4.csv", index=False)
#     return df

# # Uso
# df_github = extract_github_code_snippets(query="machine learning", language="Python", max_results=10)
# print(df_github.head())

In [19]:
# import requests
# import pandas as pd

# def fetch_wikipedia_articles(search_terms, max_results=10):
#     """Obtiene artículos de Wikipedia para términos de búsqueda específicos."""
#     url = "https://en.wikipedia.org/w/api.php"
#     articles = []

#     for term in search_terms:
#         params = {
#             "action": "query",
#             "format": "json",
#             "list": "search",
#             "srsearch": term,
#             "srlimit": max_results
#         }
#         response = requests.get(url, params=params)
#         if response.status_code == 200:
#             data = response.json()
#             for result in data["query"]["search"]:
#                 articles.append({
#                     "title": result["title"],
#                     "snippet": result["snippet"],
#                     "pageid": result["pageid"]
#                 })
#         else:
#             print(f"Error fetching articles for '{term}': {response.status_code}")

#     return pd.DataFrame(articles)

# # Guardar datos en CSV
# df_wikipedia = fetch_wikipedia_articles(["Python programming", "Machine learning"], max_results=5)
# df_wikipedia.to_csv("wikipedia_articles_dataset.csv", index=False)
# print(df_wikipedia.head())

In [20]:
# import requests

# url_github = "https://api.github.com/search/code"
# params_github = {
#     "q": "def+in:file+language:python",
#     "per_page": 100,
#     "page": 1
# }

# headers = {
#     "Accept": "application/vnd.github.v3.text-match+json",
# }

# all_data_github = []

# # Obtener datos de múltiples páginas de GitHub
# for page in range(1, 11):  # 10 páginas * 100 resultados por página = 1,000 resultados
#     params_github["page"] = page
#     try:
#         response = requests.get(url_github, params=params_github, headers=headers)
#         response.raise_for_status()  # Verificar si la solicitud fue exitosa
#         data = response.json()
#         if "items" in data:
#             all_data_github.extend(data["items"])
#             print(f"GitHub Page {page} processed")
#         else:
#             print(f"GitHub Page {page} does not contain 'items'")
#     except requests.exceptions.RequestException as e:
#         print(f"Error processing GitHub page {page}: {e}")
#     except ValueError as e:
#         print(f"Error decoding JSON for GitHub page {page}: {e}")

# # Procesar los Datos de GitHub
# texts_github = []
# labels_github = []

# for item in all_data_github:
#     code_snippets = item.get("text_matches", [{}])[0].get("fragment", "")
#     if code_snippets:
#         texts_github.append(code_snippets)
#         labels_github.append("code")
#     else:
#         texts_github.append("")
#         labels_github.append("non-code")

# # Crear el DataFrame de GitHub
# df_github = pd.DataFrame({
#     "text": texts_github,
#     "label": labels_github
# })

# # Guardar el DataFrame en un archivo CSV
# df_github.to_csv("github_code_dataset.csv", index=False)

# # Mostrar las primeras filas del DataFrame
# print(df_github.head())

In [21]:
# df_stackoverflow.label.value_counts()

In [22]:
# df_github.label.value_counts()

In [23]:
# new_data = pd.concat([df_stackoverflow, df_github], ignore_index=True)

##### Entrenamiento del modelo

In [24]:
# new_data = pd.read_csv("stackoverflow_code_dataset.csv")

In [25]:
csvs = ["stackoverflow_code_dataset.csv", "github_code_dataset.csv", "github_code_dataset_2.csv"]

new_data = pd.concat([pd.read_csv(csv) for csv in csvs], ignore_index=True)

In [26]:
new_data["label"].value_counts()

label
code        1805
non-code     805
Name: count, dtype: int64

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score

# PASO 1: Crear una matriz TF-IDF de los textos
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(new_data["text"])
y = new_data["label"]

# PASO 2: Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# PASO 3: Entrenar el Modelo
classifier = RandomForestClassifier(random_state=42, n_estimators=1000, n_jobs=-1, bootstrap=True, max_depth=None, min_samples_leaf=1, min_samples_split=2, max_features="sqrt")
classifier.fit(X_train, y_train)

# PASO 4: Evaluar el Modelo
y_pred = classifier.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# PASO 5: Función para predecir si un texto contiene código
def predict_code(text):
    text_features = vectorizer.transform([text])
    prediction = classifier.predict(text_features)
    return prediction[0]


# Ejemplo
text = "def saludo():\n    print('Hola Mundo')"
print("Prediction:", predict_code(text))

Accuracy: 0.9757343550446999
              precision    recall  f1-score   support

        code       0.98      0.98      0.98       568
    non-code       0.96      0.95      0.96       215

    accuracy                           0.98       783
   macro avg       0.97      0.97      0.97       783
weighted avg       0.98      0.98      0.98       783

Prediction: code


#### Función final

In [28]:
html_exp = re.compile(r"<.*?>")   # Detectar etiquetas HTML individuales
html_snippet_exp = re.compile(r"^\s*<.*>.*</.*>\s*$")   # Detectar fragmentos HTML completos

# Expresiones regulares para detectar sintaxis Python
python_keywords = re.compile(r"\b(def|class|import|from|return|if|elif|else|for|while|try|except|with|as|lambda|print)\b")
python_structures = re.compile(r"[{}:()=\[\]]")   # Caracteres comunes en Python
python_comments = re.compile(r"^\s*#")   # Comentarios en Python


def is_python_code(line):
    """Detecta si una línea tiene características de código Python."""

    if python_keywords.search(line) or python_structures.search(line) or python_comments.search(line):
        return True
    return False


def predict_code(linea):
    """Función de predicción que clasifica si una línea es código o no utilizando el modelo entrenado."""

    text_features = vectorizer.transform([linea])
    prediction = classifier.predict(text_features)
    
    if prediction[0] == "code" and is_python_code(linea):   # Confirmar que sea código Python
        return "code"
    return "non-code"


def code_extraction(texto: str) -> dict:
    """
    Extrae etiquetas HTML, fragmentos HTML completos y fragmentos de código de un texto.
    
    - texto: El texto a analizar.
    - return: Diccionario con 'etiquetas_html', 'fragmentos_html' y 'fragmentos_codigo'.
    """
    fragmentos_codigo = []
    fragmentos_html = []
    etiquetas_html = []

    # Procesar el texto línea por línea
    for linea in texto.splitlines():
        linea = linea.strip()
        if linea:   # Ignorar líneas vacías
            # Extraer etiquetas HTML individuales
            etiquetas = html_exp.findall(linea)
            if etiquetas:
                etiquetas_html.extend(etiquetas)
            
            # Detectar fragmentos HTML completos
            if html_snippet_exp.match(linea):
                fragmentos_html.append(linea)
            else:
                # Identificar si la línea es código
                if predict_code(linea) == "code":
                    fragmentos_codigo.append(linea)

    # Se devuelven resultados en un diccionario
    return {
        "etiquetas_html": etiquetas_html if etiquetas_html else None,
        "fragmentos_html": fragmentos_html if fragmentos_html else None,
        "fragmentos_codigo": fragmentos_codigo if fragmentos_codigo else None
    }


# Ejemplo
texto_post = """
Here is some Python code
def saludo():
    print('Hola Mundo')

Here is some HTML
<html><body>Hola Mundo</body></html>

Just some regular text here.
"""

# Ejecutar la función y mostrar resultados
resultado = code_extraction(texto_post)
print("Etiquetas HTML extraídas:", resultado.get("etiquetas_html", "No se encontraron etiquetas HTML"))
print("Fragmentos HTML extraídos:", resultado.get("fragmentos_html", "No se encontraron fragmentos HTML"))
print("Fragmentos de código extraídos:", resultado.get("fragmentos_codigo", "No se encontraron fragmentos de código"))

Etiquetas HTML extraídas: ['<html>', '<body>', '</body>', '</html>']
Fragmentos HTML extraídos: ['<html><body>Hola Mundo</body></html>']
Fragmentos de código extraídos: ['def saludo():', "print('Hola Mundo')"]
